In [1]:
import requests
import pandas as pd
import json

In [16]:
# parameters
start_date = '2025-09-25 00:00:00'
end_date = '2025-10-01 00:00:00'
tickers_list = ['BTCUSDT']
binance_url = 'https://api.binance.com'

start_date_unix_ms = int(pd.to_datetime(start_date).timestamp()*1000)
end_date_unix_ms = int(pd.to_datetime(end_date).timestamp()*1000)

In [10]:
def make_a_request(url, params):
    
    request = requests.get(url, params)
    
    return request

In [11]:
def get_binance_aggregated_historical_trades(ticker, start_time, end_time):
    
    url = binance_url + '/api/v3/aggTrades'
    params = {
        'symbol': ticker,
        'startTime': start_time,
        'endTime': end_time,
        'limit': 1000,
    }
    
    request = make_a_request(url, params)
    
    return request

In [12]:
def get_binance_data(ticker, start_time, end_time):
    
    df = pd.DataFrame()
    min_start_time = start_time
    while True:
        response = get_binance_aggregated_historical_trades(ticker, min_start_time, end_time)
        response_json = response.json()
        response_df = pd.DataFrame(response_json)
        min_start_time = int(response_df['T'].max())
        df = pd.concat([df, response_df])
        if len(response_df) < 1000:
            break
    
    #df = df[['p', 'q', 'T', 'm']]
    df = df.rename(columns={'p': 'price', 'q': 'quantity', 'T': 'unix_timestamp', 'm': 'is_buyer_maker'})
    df['price'] = df['price'].apply(lambda x: float(x))
    df['quantity'] = df['quantity'].apply(lambda x: float(x))
    df['unix_timestamp'] = df['unix_timestamp'].apply(lambda x: int(x))
    
    df['quantity_usdt'] = df['price']*df['quantity']
    df['timestamp'] = df['unix_timestamp'].apply(lambda x: pd.to_datetime(x, unit='ms'))
    df['taker_side'] = df['is_buyer_maker'].apply(lambda x: 'buy' if x == False else 'sell')
    
    df = df.drop_duplicates().reset_index(drop=True)
    df = df[['timestamp', 'price', 'quantity', 'taker_side', 'quantity_usdt']] # , 'unix_timestamp'
    
    return df

In [17]:
df = get_binance_data(tickers_list[0], start_date_unix_ms, end_date_unix_ms)

In [18]:
df

,timestamp,price,quantity,taker_side,quantity_usdt
0,2025-09-25 00:00:00.010,113307.01,0.03065,buy,3472.859856
1,2025-09-25 00:00:00.198,113307.01,0.26476,buy,29999.163968
2,2025-09-25 00:00:00.221,113307.00,0.00142,sell,160.895940
3,2025-09-25 00:00:00.330,113307.00,0.00034,sell,38.524380
4,2025-09-25 00:00:00.371,113307.01,0.02647,buy,2999.236555
...,...,...,...,...,...
4537498,2025-09-30 23:59:57.787,114048.94,0.00215,buy,245.205221
4537499,2025-09-30 23:59:58.491,114048.94,0.00059,buy,67.288875
4537500,2025-09-30 23:59:58.613,114048.94,0.00033,buy,37.636150
4537501,2025-09-30 23:59:58.734,114048.93,0.00027,sell,30.793211


In [19]:
df.to_pickle("./raw_data_binance_BTCUSDT.pkl")